<a href="https://colab.research.google.com/github/sudharshan-saranathan/pdf-parser/blob/main/Chart_Pipeline_Demo_Blank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chart Understanding Pipeline — Build From Scratch

**VelTech FDP Workshop**

Implement each stage of the pipeline using the pre-trained models and libraries provided.

| Step | Task |
|------|------|
| 1 | Install dependencies |
| 2 | Download weights + sample PDFs |
| 3 | Rasterize PDF pages to images |
| 4 | Detect chart regions (YOLO) |
| 5 | Classify chart type (YOLO-cls) |
| 6 | OCR each crop (PaddleOCR) |
| 7 | Assign semantic roles (Qwen2.5-VL) |
| 8 | Wire full pipeline |


## STEP 1 — Install dependencies

In [ ]:
# ==== PADDLEOCR CPU INSTALL : SAFE COLAB BLOCK ====

!python -V
!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr paddlex || true
!pip install -U pip setuptools wheel

# Official Paddle CPU wheel index shown in PaddleOCR quick start
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

# Install PaddleOCR after PaddlePaddle
!python -m pip install paddleocr

#RESTART THE SESSION

Python 3.12.13
Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cpu/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.0/189.0 MB 43.2 MB/s  0:00:04
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [paddlepaddle]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 15.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 21.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 37.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 78.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 118.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 86.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 39.1 MB/s  0:00:00
  Attempting uninstall: PyYAML
    Found existing installation: PyYAML 6

In [ ]:
# After Restarting the Session Run From Here
#==== CLEAN CONFLICTS ====

!pip uninstall -y \
  langchain \
  langchain-core \
  langchain-community \
  langchain-text-splitters \
  langsmith \
  paddleocr \
  paddlex \
  paddlepaddle \
  paddlepaddle-gpu

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.3.1
Uninstalling langchain-core-1.3.1:
  Successfully uninstalled langchain-core-1.3.1
Found existing installation: langsmith 0.7.34
Uninstalling langsmith-0.7.34:
  Successfully uninstalled langsmith-0.7.34
Found existing installation: paddleocr 3.5.0
Uninstalling paddleocr-3.5.0:
  Successfully uninstalled paddleocr-3.5.0
Found existing installation: paddlex 3.5.2
Uninstalling paddlex-3.5.2:
  Successfully uninstalled paddlex-3.5.2
Found existing installation: paddlepaddle 3.2.0
Uninstalling paddlepaddle-3.2.0:
  Successfully uninstalled paddlepaddle-3.2.0


In [ ]:
# ==== INSTALL PADDLE OCR (CPU SAFE) ====

!python -V
!pip install -U pip setuptools wheel

# Official Paddle CPU install
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

# Then PaddleOCR
!python -m pip install paddleocr==3.4.0

Python 3.12.13
Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cpu/


In [ ]:
# ==== Qwen2.5-VL ====
!pip install -q transformers accelerate qwen-vl-utils

# ==== YOLO + PDF + Drive download ====
!pip install -q ultralytics opencv-python pillow matplotlib pandas tqdm
!pip install -q pymupdf gdown

print("✅ Other dependencies installed")
!nvidia-smi | head -10

✅ Other dependencies installed
/bin/bash: line 1: nvidia-smi: command not found


## STEP 2 — Download weights and sample PDFs

All model weights and sample PDFs are available via the links below.

In [10]:
# ── Config: all external links in one place ───────────────────────────────
DRIVE_IDS = {
    "demo_bundle":       "YOUR_BUNDLE_DRIVE_ID",
    "sample_papers":     "1m4MgUIP93ds8Jr7lMyp6csSKPPneew5w",
}

WEIGHT_FILES = {
    "chart_detector":    "chart_detector_v3.pt",
    "plot_classifier":   "Plot_Classifier_new_latest.pt",
    "bar_detection":     "Bar_Detection_Yolo_v4.pt",
    "line_plot_area":    "plot_area_detector_line_v1.pt",
    "line_embed":        "best_line_embed_instance_v1.pt",
}

# Tunable parameters
PAGE_ZOOM            = 2.0
DETECT_CONF          = 0.25
DETECT_IOU           = 0.45
CROP_PAD             = 12
CLASSIFIER_MIN_CONF  = 0.50
OCR_MIN_CONF         = 0.60
MAX_OCR_BOXES        = 18


In [ ]:
# Path-semantics
from pathlib import Path
import zipfile
import shutil

# Define ROOT and ZIPP
ROOT = Path("/content/chart_demo")
ZIPP = Path("/content/chart_bar_line_qwen_demo_bundle.zip")

# Demo bundle Google Drive ID (replace with your own if you re-package)
DRIVE_FILE_ID = "185upEGS0nI_IjGxoneglgjIfKBGvRNnU"

if ROOT.exists():
    shutil.rmtree(ROOT) # Remove ROOT dir if it already exists

ROOT.mkdir(parents=True, exist_ok=True)

!gdown --id "$DRIVE_FILE_ID" -O "$ZIPP" # Download

with zipfile.ZipFile(ZIPP, "r") as z:
    z.extractall(ROOT)

# Normalize any Windows-style backslash paths inside the zip
for p in list(ROOT.rglob("*")):
    if p.is_file() and "\\" in p.name:
        parts = p.name.split("\\")
        new_path = p.parent.joinpath(*parts)
        new_path.parent.mkdir(parents=True, exist_ok=True)
        if not new_path.exists():
            shutil.move(str(p), str(new_path))

print("✅ Extracted to:", ROOT)
for p in sorted(ROOT.iterdir())[:15]:
    print(" -", p.name)

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=185upEGS0nI_IjGxoneglgjIfKBGvRNnU
From (redirected): https://drive.google.com/uc?id=185upEGS0nI_IjGxoneglgjIfKBGvRNnU&confirm=t&uuid=9feafd46-c349-47f9-85be-734603abff25
To: /content/chart_bar_line_qwen_demo_bundle.zip
100% 199M/199M [00:01<00:00, 120MB/s]
✅ Extracted to: /content/chart_demo
 - README_WEIGHTS.txt
 - weights


In [14]:
# Unzip the sample papers to the require directory

from pathlib import Path
import shutil, json, zipfile
import fitz  # PyMuPDF
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path("/content/chart_demo")
SAVE  = Path("/content/chart_demo_outputs")
PAGE_OUT_DIR = SAVE / "pdf_pages"

# ── Pre-filled: download sample papers and locate weights ─────────────────
SAMPLE_PAPERS_DRIVE_ID = "1m4MgUIP93ds8Jr7lMyp6csSKPPneew5w"
SAMPLE_DIR = ROOT / "sample_papers"
SAMPLE_ZIP = Path("/content/sample_papers.zip")

# Unzip the SAMPLE_ZIP to inspect PDFs
if not SAMPLE_DIR.exists():
    !gdown --id "$SAMPLE_PAPERS_DRIVE_ID" -O "$SAMPLE_ZIP" # Download

zipfile.ZipFile(SAMPLE_ZIP, "r").extractall(SAMPLE_DIR)

# List PDFs
print("Sample PDFs:")
for p in sorted(SAMPLE_DIR.glob("*.pdf")):
    print(" -", p.name)

Sample PDFs:
 - W08-1103.pdf
 - W11-0506.pdf
 - fcomp-03-706939.pdf
 - feduc-10-1639528.pdf
 - feduc-11-1780665.pdf
 - sustainability-16-03389.pdf


## STEP 3 — Rasterize PDF pages to images

In [11]:
# Open the PDF and convert every page to a PNG image.
# Save to /content/chart_demo_outputs/pdf_pages/
# Libraries: fitz (PyMuPDF), pathlib

## STEP 4 — Detect chart regions and crop

In [ ]:
# For each page image, run the chart detector (YOLO).
# Crop each detected region (add CROP_PAD padding).
# Save crops to /content/chart_demo_outputs/chart_crops/
# Save metadata to chart_crops_metadata.csv


## STEP 5 — Classify chart type

In [ ]:
# For each crop, run the plot classifier (YOLO-cls).
# Record pred_chart_type and pred_chart_type_confidence.
# Save to chart_crops_classified_metadata.csv


## STEP 6 — OCR each crop

In [ ]:
# Run PaddleOCR on each crop.
# For each detected text region, extract:
#   text, bounding box, confidence, angle
# Featurize each box: position band, numeric/year/rotated flags.


## STEP 7 — Assign semantic roles with Qwen2.5-VL

In [ ]:
# Load Qwen2.5-VL. For each crop, send the OCR boxes + image to Qwen.
# Ask it to assign a role to each box:
#   title, x_tick, y_tick, data_label, legend, caption, other
# Parse the JSON response and attach roles to boxes.


## STEP 8 — Wire the full pipeline

In [ ]:
# For each bar/line crop:
#   A. Run OCR + featurize
#   B. Assign roles via Qwen
#   C. Draw coloured bounding boxes and save overlay
#   D. Run the YOLO extractor (bar_det or line_plot_area)
# Display results in a grid.
